# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published on: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets and field information.

All record sets, fields, and columns are referenced by their `@id` fields as specified in the Croissant metadata.

In [ ]:
recordsets = list(dataset.record_sets)
print(f"Total number of record sets: {len(recordsets)}\n")

for rs in recordsets:
    print(f"RecordSet '@id': {rs['@id']}")
    print(f"  Name: {rs.get('name', 'Unknown')}")
    print(f"  Description: {rs.get('description', 'N/A')}")
    # List the fields for this record set
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Fields:")
    for field in fields:
        # field may be string (the @id) or object
        if isinstance(field, str):
            print(f"    - Field '@id': {field}")
        else:
            print(f"    - Field '@id': {field.get('@id','?')} (name: {field.get('name','?')})")
    print()

## 3. Data Extraction
Load data from record sets into pandas DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# We'll extract all available record sets and show the main one.
record_set_ids = [rs['@id'] for rs in recordsets]
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded RecordSet '@id': {rs_id} with shape {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load records for RecordSet '@id': {rs_id}. Error: {e}")

# For demonstration, pick the first record set for further steps
if dataframes:
    main_rs_id = record_set_ids[0]
    print(f"\nMain record set chosen for EDA: {main_rs_id}\n")
    print(dataframes[main_rs_id].head())
else:
    main_rs_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. All entity IDs (fields and record sets) are referenced using their `@id`.

In [ ]:
# Select a numeric field for analysis (chosen by inspection of columns)
if main_rs_id:
    df = dataframes[main_rs_id]
    # Try to identify numeric columns
    numeric_cols = [col for col in df.columns if np.issubdtype(df[col].dropna().apply(type).mode().values[0], np.number)]
    # If none found, try to coerce columns to numeric
    if not numeric_cols and not df.empty:
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if df[col].notna().any():
                    numeric_cols.append(col)
            except Exception:
                continue

    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        print(f"Using numeric field for EDA: '{numeric_field_id}'")
        threshold = df[numeric_field_id].dropna().median()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to group by a likely categorical field
        candidate_group_fields = [col for col in df.columns if col != numeric_field_id]
        group_field_id = None
        for col in candidate_group_fields:
            if df[col].dtype == object and df[col].nunique() < min(10, len(df)):
                group_field_id = col
                break
        if group_field_id:
            print(f"\nGrouping by: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(grouped_df.head())
    else:
        print("No numeric fields found to analyze.")
else:
    print("No main record set DataFrame loaded.")

## 5. Visualization
Visualize distribution for the selected numeric field and relationship with the group field (if applicable).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_cols:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouping field is present, show boxplot
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"Distribution of {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we've loaded and explored the 'Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells' dataset using the `mlcroissant` library, referencing all entities by their `@id`. 
- We obtained metadata, located record sets and their fields, and loaded the main record set into a DataFrame.
- Exploratory analysis included filtering and normalizing a numeric field, grouping by categorical fields, and basic statistical visualizations.

This workflow provides a foundation for further, domain-specific analyses—such as protein biomarker discovery or cross-sample comparison—using reproducible and FAIR-aligned pipelines.